**Comparative Evaluation: Greedy_SOSP vs Robust_MOSP vs Dijkstra**

This notebook benchmarks `Greedy_SOSP`, `Robust_MOSP`, and a baseline `Dijkstra` implementation across varying graph sizes and edge densities. 

In [1]:
import time
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from mpl_toolkits.mplot3d import Axes3D

from Greedy_SOSP import Greedy_SOSP
from Robust_MOSP import Robust_MOSP
from utils.generate_graph import generate_graph

NODE_SIZES = [4, 8, 16, 32, 64, 128, 256, 512, 1024]
EDGE_PROBS = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
REPEAT = 10

def dijkstra_baseline_runtime(G, source=None):
    if source is None:
        source = next(iter(G.nodes()))
    start = time.perf_counter()
    nx.single_source_dijkstra_path_length(G, source, weight="expected_energy")
    return time.perf_counter() - start

def run_comparison_trial(n, p, repeat_id=None):
    G = generate_graph(n, p, seed=repeat_id, with_metrics=True)
    source = next(iter(G.nodes()))

    start = time.perf_counter()
    Greedy_SOSP(G)
    greedy_time = time.perf_counter() - start

    start = time.perf_counter()
    Robust_MOSP(G)
    robust_time = time.perf_counter() - start

    dijkstra_time = dijkstra_baseline_runtime(G, source)

    return {
        "requested_nodes": n,
        "actual_nodes": G.number_of_nodes(),
        "edge_prob": p,
        "repeat": repeat_id,
        "greedy_runtime": greedy_time,
        "robust_runtime": robust_time,
        "dijkstra_runtime": dijkstra_time,
    }

In [ ]:
results = []
total_runs = len(NODE_SIZES) * len(EDGE_PROBS) * REPEAT

with tqdm(total=total_runs, desc="Benchmarking") as pbar:
    for n in NODE_SIZES:
        for p in EDGE_PROBS:
            for r in range(REPEAT):
                trial = run_comparison_trial(n, p, repeat_id=r)
                results.append(trial)
                pbar.update(1)

df_comparison = pd.DataFrame(results)
df_comparison.to_csv("Greedy_Robust_Dijkstra_Comparison_results.csv", index=False)
df_comparison.head()

**df_comparison results can be further examined extensively for research purposes**